# SSP5-8.5 NC File Investigation
## Notebook 18

## 1. Imports

In [1]:
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path

print(f"xarray  : {xr.__version__}")
print(f"numpy   : {np.__version__}")
print(f"pandas  : {pd.__version__}")

xarray  : 2025.12.0
numpy   : 2.1.3
pandas  : 2.2.3


## 2. Paths

In [2]:
MODEL_DIR_85 = Path(r"D:\RICAAR8.5\merge\Precipitation")

OUTPUT_ROOT  = Path(r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\ssp5-8.5 output")
for sub in ["3 ensemble models", "6 ensemble models",
            r"projections\seasonal_means",
            r"projections\differences",
            r"projections\figures"]:
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)

print(f"Model dir : {MODEL_DIR_85}")
print(f"Output    : {OUTPUT_ROOT}")
print(f"Exists    : {MODEL_DIR_85.exists()}")

Model dir : D:\RICAAR8.5\merge\Precipitation
Output    : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\ssp5-8.5 output
Exists    : True


## 3. List All NC Files in the Directory

In [3]:
nc_files = sorted(MODEL_DIR_85.glob("*.nc"))

print(f"NC files found: {len(nc_files)}")
print()
print(f"{'Filename':<45} {'Size (MB)':>10}")
print("-" * 57)
for f in nc_files:
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:<43} {size_mb:>10.1f}")

NC files found: 6

Filename                                       Size (MB)
---------------------------------------------------------
  merged_CMCC-CM2-SR5.nc                           981.9
  merged_CNRM-ESM2-1.nc                            981.9
  merged_EC-Earth3-Veg.nc                          981.9
  merged_IPSL-CM6A-LR.nc                           981.9
  merged_MPI-ESM1-2-LR.nc                          981.9
  merged_NorESM2-MM.nc                             981.9


## 4. Inspect Each File — Grid, Time, Variables, Value Ranges

In [4]:
# Reference SSP2-4.5 specs to compare against
REF_NLAT   = 85
REF_NLON   = 75
REF_VAR    = "prAdjust"
REF_TSTART = "1961-01-01"
REF_TEND   = "2070-12-31"

print("=" * 75)
print("DETAILED FILE INSPECTION")
print("=" * 75)

summary_rows = []

for f in nc_files:
    print(f"\nFile: {f.name}")
    print("-" * 60)

    ds = xr.open_dataset(f)

    # Dimensions
    print(f"  Dimensions : {dict(ds.dims)}")

    # Coordinates
    print(f"  Coordinates: {list(ds.coords)}")

    # Variables
    print(f"  Variables  : {list(ds.data_vars)}")

    # Lat / lon ranges
    lat_name = "lat" if "lat" in ds.coords else "latitude"
    lon_name = "lon" if "lon" in ds.coords else "longitude"
    lats = ds[lat_name].values
    lons = ds[lon_name].values
    print(f"  Lat range  : {lats.min():.2f} – {lats.max():.2f}°N  ({len(lats)} points)")
    print(f"  Lon range  : {lons.min():.2f} – {lons.max():.2f}°E  ({len(lons)} points)")

    # Time range
    times = ds["time"].values
    t0    = pd.Timestamp(times[0]).date()
    t1    = pd.Timestamp(times[-1]).date()
    print(f"  Time range : {t0} → {t1}  ({len(times):,} steps)")

    # Precipitation variable stats (first available)
    pr_var = None
    for candidate in ["prAdjust", "pr", "precipitation", "precip", "tp"]:
        if candidate in ds.data_vars:
            pr_var = candidate
            break
    if pr_var:
        # Sample from a single year to keep it fast
        da_sample = ds[pr_var].sel(time=slice("2000-01-01", "2000-12-31")).values
        da_sample = da_sample[~np.isnan(da_sample)]
        print(f"  Precip var : {pr_var}")
        print(f"  Attrs      : {ds[pr_var].attrs}")
        print(f"  2000 stats : mean={da_sample.mean():.4f}  "
              f"max={da_sample.max():.4f}  min={da_sample.min():.4f}")
    else:
        print("  Precip var : NOT FOUND")

    # Grid match check
    grid_ok = (len(lats) == REF_NLAT) and (len(lons) == REF_NLON)
    var_ok   = (pr_var == REF_VAR)
    time_ok  = (str(t0) == REF_TSTART) and (str(t1) == REF_TEND)
    print(f"  Grid match : {'✓' if grid_ok else '✗ MISMATCH'}  "
          f"Var match: {'✓' if var_ok else f'✗ ({pr_var})'}  "
          f"Time match: {'✓' if time_ok else f'✗ ({t0} → {t1})'}")

    summary_rows.append({
        "File"      : f.name,
        "n_lat"     : len(lats),
        "n_lon"     : len(lons),
        "n_time"    : len(times),
        "t_start"   : str(t0),
        "t_end"     : str(t1),
        "pr_var"    : pr_var,
        "grid_ok"   : grid_ok,
        "var_ok"    : var_ok,
        "time_ok"   : time_ok,
    })

    ds.close()

print()
print("=" * 75)
print("SUMMARY TABLE")
print("=" * 75)
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

DETAILED FILE INSPECTION

File: merged_CMCC-CM2-SR5.nc
------------------------------------------------------------


C:\Users\ASUS\AppData\Local\Temp\ipykernel_30520\2206178462.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dimensions : {dict(ds.dims)}")
C:\Users\ASUS\AppData\Local\Temp\ipykernel_30520\2206178462.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dimensions : {dict(ds.dims)}")
C:\Users\ASUS\AppData\Local\Temp\ipykernel_30520\2206178462.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, ple

  Dimensions : {'time': 40177, 'bnds': 2, 'lat': 85, 'lon': 75}
  Coordinates: ['lon', 'lat', 'time']
  Variables  : ['time_bnds', 'prAdjust']
  Lat range  : 27.05 – 35.45°N  (85 points)
  Lon range  : 33.55 – 40.95°E  (75 points)
  Time range : 1961-01-01 → 2070-12-31  (40,177 steps)
  Precip var : prAdjust
  Attrs      : {'standard_name': 'precipitation_flux', 'long_name': 'Precipitation', 'units': 'mm/day', 'cell_methods': 'time: mean'}
  2000 stats : mean=0.3452  max=90.9227  min=0.0000
  Grid match : ✓  Var match: ✓  Time match: ✓

File: merged_CNRM-ESM2-1.nc
------------------------------------------------------------
  Dimensions : {'time': 40177, 'bnds': 2, 'lat': 85, 'lon': 75}
  Coordinates: ['lon', 'lat', 'time']
  Variables  : ['time_bnds', 'prAdjust']
  Lat range  : 27.05 – 35.45°N  (85 points)
  Lon range  : 33.55 – 40.95°E  (75 points)
  Time range : 1961-01-01 → 2070-12-31  (40,177 steps)
  Precip var : prAdjust
  Attrs      : {'standard_name': 'precipitation_flux', 'lo

C:\Users\ASUS\AppData\Local\Temp\ipykernel_30520\2206178462.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dimensions : {dict(ds.dims)}")
C:\Users\ASUS\AppData\Local\Temp\ipykernel_30520\2206178462.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dimensions : {dict(ds.dims)}")


## 5. Confirm Model-to-Filename Mapping

In [5]:
MODELS = [
    "CMCC-CM2-SR5",
    "CNRM-ESM2-1",
    "EC-Earth3-Veg",
    "IPSL-CM6A-LR",
    "MPI-ESM1-2-LR",
    "NorESM2-MM",
]

print(f"{'Model':<20} {'Expected filename':<40} {'Found':>6}")
print("-" * 68)
all_found = True
for model in MODELS:
    expected = MODEL_DIR_85 / f"merged_{model}.nc"
    found    = expected.exists()
    if not found:
        all_found = False
    print(f"  {model:<18} {'merged_'+model+'.nc':<40} {'✓' if found else '✗ MISSING'}")

print()
if all_found:
    print("All 6 model files confirmed. Ready to proceed to NB19.")
else:
    print("WARNING: Some files missing — check filenames above before proceeding.")

Model                Expected filename                         Found
--------------------------------------------------------------------
  CMCC-CM2-SR5       merged_CMCC-CM2-SR5.nc                   ✓
  CNRM-ESM2-1        merged_CNRM-ESM2-1.nc                    ✓
  EC-Earth3-Veg      merged_EC-Earth3-Veg.nc                  ✓
  IPSL-CM6A-LR       merged_IPSL-CM6A-LR.nc                   ✓
  MPI-ESM1-2-LR      merged_MPI-ESM1-2-LR.nc                  ✓
  NorESM2-MM         merged_NorESM2-MM.nc                     ✓

All 6 model files confirmed. Ready to proceed to NB19.


## 6. Verify Grid Matches SSP2-4.5 (Coordinate Comparison)

In [6]:
# Compare lat/lon arrays between SSP2-4.5 and SSP5-8.5 for the same model
SSP245_DIR = Path(
    r"D:\RICAAR\riccar-data_jordan-ssp2-4-5-daily-data_2024-07-29_0915"
    r"\Merge\Precipitation"
)

model_test = "CMCC-CM2-SR5"
f245 = SSP245_DIR / f"{model_test}.nc"
f585 = MODEL_DIR_85 / f"merged_{model_test}.nc"

ds245 = xr.open_dataset(f245)
ds585 = xr.open_dataset(f585)

lats245 = ds245["lat"].values
lons245 = ds245["lon"].values
lats585 = ds585["lat"].values
lons585 = ds585["lon"].values

lat_match = np.allclose(lats245, lats585, atol=1e-4)
lon_match = np.allclose(lons245, lons585, atol=1e-4)

print(f"SSP2-4.5 grid : {len(lats245)} lat × {len(lons245)} lon")
print(f"SSP5-8.5 grid : {len(lats585)} lat × {len(lons585)} lon")
print()
print(f"Lat arrays match : {'✓ YES' if lat_match else '✗ NO — grids differ!'}")
print(f"Lon arrays match : {'✓ YES' if lon_match else '✗ NO — grids differ!'}")

if not lat_match or not lon_match:
    print()
    print("SSP2-4.5 lats:", lats245[:5], "...", lats245[-5:])
    print("SSP5-8.5 lats:", lats585[:5], "...", lats585[-5:])

ds245.close()
ds585.close()

print()
if lat_match and lon_match:
    print("Grids are identical. The basin_id grid from NB05 can be reused directly in NB19/NB20.")
else:
    print("WARNING: Grids differ — basin_id grid must be rebuilt for SSP5-8.5 data.")

SSP2-4.5 grid : 85 lat × 75 lon
SSP5-8.5 grid : 85 lat × 75 lon

Lat arrays match : ✓ YES
Lon arrays match : ✓ YES

Grids are identical. The basin_id grid from NB05 can be reused directly in NB19/NB20.
